In [3]:
#in_dir = "/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S/_25_07_DRUM/d_40_percent_silence"
in_dir  ="/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S"
out_dir = "/Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_2_SNARE"

# GET ONE SNARE SAMPLE 

In [4]:
# =========================================================
# -----######-----######  CORE IMPORTABLE FUNCTION  ######-----######-----######
# =========================================================

import os
import re
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm


def _snare_0403_i1_GET_df_snareonly_mp3_from_folder(
    in_dir,
    out_dir,
    audio_extensions=(".mp3", ".MP3"),
    # audio
    sr_target=44100,
    hop_length=256,
    n_fft=2048,
    # filename BPM parse (used only as a safety cap)
    bpm_cap_beats=0.75,          # snares can be up to ~3/4 beat if roomy; set 0.5 for tighter
    # start detection
    pre_ms=5,
    min_gap_ms=90,
    # snare specificity
    snare_band=(1800, 8000),     # snare "crack" / noise body
    low_reject_band=(30, 180),   # reject kick
    hat_reject_band=(9000, 16000),# reject pure hats/air
    snare_over_low_ratio=4.0,    # must beat the low band clearly
    snare_over_hat_ratio=1.25,   # must have more body than pure air
    rms_gate_db=-48,
    peak_gate_db=-24,
    centroid_min_hz=2200,        # snares are bright-ish but not pure hat
    centroid_max_hz=9000,        # reject hats
    # tail trimming (variable length)
    min_snare_ms=60,
    max_snare_ms=420,
    end_hold_ms=18,
    end_db_drop=20,
    end_floor_db=-60,
    fade_ms=6,
    # mp3 export
    mp3_bitrate="320k",
    ffmpeg_path="ffmpeg",
    overwrite=True,
    max_files=None,
):
    """
    Recursively find all MP3 files.
    For each MP3:
      - Parse BPM from filename: last '-<BPM>.mp3' (used only for max duration cap)
      - Detect ONE best snare-like transient:
          * dominant 1.8k–8k energy (crack/body)
          * reject low-end dominance (kick)
          * reject ultra-high dominance (hat-only)
          * spectral centroid within snare-ish range
      - Trim tail using snare-band decay => "just the snare", variable length
      - Export as MP3 with SAME filename into out_dir
      - Originals untouched

    Returns:
      df_out: one row per mp3 with timings + export path + debug stats
    """

    os.makedirs(out_dir, exist_ok=True)

    if shutil.which(ffmpeg_path) is None:
        raise RuntimeError(
            f"ffmpeg not found on PATH as '{ffmpeg_path}'. "
            "Install via: brew install ffmpeg  (or pass ffmpeg_path)"
        )

    # gather files recursively
    exts = tuple(audio_extensions) if isinstance(audio_extensions, (list, tuple)) else (audio_extensions,)
    mp3_paths = []
    for root, _, files in os.walk(in_dir):
        for fn in files:
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            if fn.endswith(exts):
                mp3_paths.append(os.path.join(root, fn))

    mp3_paths = sorted(mp3_paths)
    if max_files is not None:
        mp3_paths = mp3_paths[: int(max_files)]

    # bpm parser: last "-<bpm>.mp3"
    re_bpm = re.compile(r"-([0-9]+(?:\.[0-9]+)?)\.mp3$", re.IGNORECASE)

    def _parse_bpm_from_name(path):
        base = os.path.basename(path)
        m = re_bpm.search(base)
        if not m:
            return None
        bpm = float(m.group(1))
        if bpm <= 0:
            return None
        return bpm

    def _peak_db(y_seg):
        pk = float(np.max(np.abs(y_seg)) + 1e-12)
        return float(20 * np.log10(pk))

    def _band_rms_db(S_mag, freqs, fr0, fr1, f0, f1):
        if fr1 <= fr0:
            return -120.0
        fmask = (freqs >= f0) & (freqs <= f1)
        band = S_mag[fmask, fr0:fr1]
        if band.size == 0:
            return -120.0
        val = float(np.sqrt(np.mean(band ** 2)) + 1e-12)
        return float(20 * np.log10(val))

    def _band_env_db(y_seg, sr, f0, f1):
        S = np.abs(librosa.stft(y_seg, n_fft=n_fft, hop_length=hop_length)) + 1e-12
        freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
        fmask = (freqs >= f0) & (freqs <= f1)
        band_energy = np.sqrt(np.mean(S[fmask, :] ** 2, axis=0)) + 1e-12
        band_db = librosa.amplitude_to_db(band_energy, ref=np.max)
        return band_db

    rows = []

    for src_path in tqdm(mp3_paths, desc="TQM | mp3 → find 1 snare → trim → export mp3", leave=True):
        base_fn = os.path.basename(src_path)
        out_path = os.path.join(out_dir, base_fn)

        try:
            bpm = _parse_bpm_from_name(src_path)
            beat_sec = (60.0 / bpm) if bpm else None

            # duration caps
            max_len_sec = max_snare_ms / 1000.0
            if beat_sec:
                max_len_sec = min(max_len_sec, bpm_cap_beats * beat_sec)

            pre_s = pre_ms / 1000.0
            min_len_sec = min_snare_ms / 1000.0

            # load mono
            y, sr = librosa.load(src_path, sr=sr_target, mono=True)
            if y is None or len(y) < int(sr * 0.25):
                raise ValueError("Audio too short or unreadable.")

            # onset detection
            onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
            onset_frames = librosa.onset.onset_detect(
                onset_envelope=onset_env,
                sr=sr,
                hop_length=hop_length,
                backtrack=False,
                pre_max=5, post_max=5, pre_avg=5, post_avg=5,
                delta=0.18, wait=0
            )
            onset_times = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)
            if len(onset_times) == 0:
                raise ValueError("No onsets detected.")

            # merge close hits
            min_gap_sec = min_gap_ms / 1000.0
            merged = []
            for t in onset_times:
                t = float(t)
                if not merged or (t - merged[-1]) >= min_gap_sec:
                    merged.append(t)

            # global STFT for band scoring
            S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

            # RMS gate
            rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
            rms_db = librosa.amplitude_to_db(rms + 1e-12, ref=np.max)

            # spectral centroid
            cent = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)[0]

            best = None

            for t in merged:
                start_sec = max(0.0, t - pre_s)
                end_sec_tmp = min(len(y) / sr, start_sec + max_len_sec)

                a0 = int(start_sec * sr)
                a1 = int(end_sec_tmp * sr)
                if a1 <= a0 + int(min_len_sec * sr):
                    continue

                # local gates
                fr = int((t * sr) / hop_length)
                fr0g = max(0, fr - 2)
                fr1g = min(len(rms_db), fr + 3)
                local_rms_db = float(np.max(rms_db[fr0g:fr1g]))

                pk_win = min(len(y), a0 + int(0.08 * sr))
                pk_db = _peak_db(y[a0:pk_win])

                if local_rms_db < rms_gate_db:
                    continue
                if pk_db < peak_gate_db:
                    continue

                # centroid window
                c0 = max(0, fr - 2)
                c1 = min(len(cent), fr + 3)
                local_cent_hz = float(np.max(cent[c0:c1]))
                if not (centroid_min_hz <= local_cent_hz <= centroid_max_hz):
                    continue

                # short window for snare signature
                short_end = min(len(y) / sr, start_sec + 0.12)
                a1s = int(short_end * sr)

                fr0b = max(0, int(a0 / hop_length))
                fr1b = min(S.shape[1], int(a1s / hop_length) + 1)

                sn_db = _band_rms_db(S, freqs, fr0b, fr1b, snare_band[0], snare_band[1])
                lo_db = _band_rms_db(S, freqs, fr0b, fr1b, low_reject_band[0], low_reject_band[1])
                ht_db = _band_rms_db(S, freqs, fr0b, fr1b, hat_reject_band[0], hat_reject_band[1])

                sn_lin = 10 ** (sn_db / 20.0)
                lo_lin = 10 ** (lo_db / 20.0)
                ht_lin = 10 ** (ht_db / 20.0)

                ratio_sn_low = float(sn_lin / (lo_lin + 1e-12))
                ratio_sn_hat = float(sn_lin / (ht_lin + 1e-12))

                if ratio_sn_low < snare_over_low_ratio:
                    continue
                if ratio_sn_hat < snare_over_hat_ratio:
                    continue

                # score: body+crack, not too airy
                score = float((ratio_sn_low * 1.1) + (ratio_sn_hat * 0.8) + (pk_db / 10.0) + (local_rms_db / 12.0))

                if (best is None) or (score > best["score"]):
                    best = {
                        "onset_sec": t,
                        "start_sec": start_sec,
                        "score": score,
                        "ratio_sn_low": ratio_sn_low,
                        "ratio_sn_hat": ratio_sn_hat,
                        "centroid_hz": local_cent_hz,
                        "peak_db_first80ms": pk_db,
                    }

            if best is None:
                raise ValueError("No snare-like onset passed filters (try loosening ratios/gates).")

            # ---- trim end via snare-band decay ----
            start_sec = best["start_sec"]
            start_samp = int(start_sec * sr)

            max_end_sec = min(len(y) / sr, start_sec + max_len_sec)
            max_end_samp = int(max_end_sec * sr)

            seg = y[start_samp:max_end_samp].copy()
            if len(seg) < int(min_len_sec * sr):
                raise ValueError("Segment too short after start selection.")

            sn_env_db = _band_env_db(seg, sr, snare_band[0], snare_band[1])  # 0 at peak

            hold_frames = max(1, int((end_hold_ms / 1000.0) * sr / hop_length))
            peak_frame = int(np.argmax(sn_env_db))

            drop_thr = -abs(end_db_drop)
            floor_thr = float(end_floor_db)

            min_end_samp = int(min_len_sec * sr)
            min_end_frame = max(0, int(min_end_samp / hop_length))

            end_frame = None
            for fr in range(max(min_end_frame, peak_frame + 1), len(sn_env_db) - hold_frames):
                window = sn_env_db[fr : fr + hold_frames]
                if np.all(window <= drop_thr) or np.all(window <= floor_thr):
                    end_frame = fr
                    break

            if end_frame is None:
                fallback_sec = min(max_len_sec, 0.20)  # typical snare tail
                end_samp_local = int(fallback_sec * sr)
            else:
                end_samp_local = int(end_frame * hop_length)

            end_samp_local = max(end_samp_local, int(min_len_sec * sr))
            end_samp_local = min(end_samp_local, len(seg))

            clip = seg[:end_samp_local].copy()

            # fade out
            fade_len = int((fade_ms / 1000.0) * sr)
            if len(clip) > fade_len + 4:
                fade = np.linspace(1.0, 0.0, fade_len)
                clip[-fade_len:] *= fade

            # export mp3 via ffmpeg
            with tempfile.TemporaryDirectory() as td:
                tmp_wav = os.path.join(td, "tmp.wav")
                sf.write(tmp_wav, clip, sr)

                cmd = [
                    ffmpeg_path, "-y" if overwrite else "-n",
                    "-i", tmp_wav,
                    "-vn",
                    "-ar", str(sr),
                    "-ac", "1",
                    "-b:a", mp3_bitrate,
                    out_path
                ]
                p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
                if p.returncode != 0:
                    raise RuntimeError(f"ffmpeg failed: {p.stderr[-600:]}")

            rows.append({
                "src_path": src_path,
                "src_file": base_fn,
                "bpm": bpm,
                "beat_sec": beat_sec,
                "onset_sec": best["onset_sec"],
                "start_sec": best["start_sec"],
                "dur_ms": float(len(clip) / sr * 1000.0),
                "score": best["score"],
                "ratio_sn_low": best["ratio_sn_low"],
                "ratio_sn_hat": best["ratio_sn_hat"],
                "centroid_hz": best["centroid_hz"],
                "out_path": out_path,
                "error": None
            })

        except Exception as e:
            rows.append({
                "src_path": src_path,
                "src_file": base_fn,
                "bpm": None,
                "beat_sec": None,
                "onset_sec": None,
                "start_sec": None,
                "dur_ms": None,
                "score": None,
                "ratio_sn_low": None,
                "ratio_sn_hat": None,
                "centroid_hz": None,
                "out_path": None,
                "error": str(e)
            })

    return pd.DataFrame(rows)

In [5]:


audio_extensions = (".mp3", ".MP3")

df_snares = _snare_0403_i1_GET_df_snareonly_mp3_from_folder(
    in_dir=in_dir,
    out_dir=out_dir,
    audio_extensions=audio_extensions,
    sr_target=44100,
    # start here (balanced)
    snare_over_low_ratio=4.0,
    snare_over_hat_ratio=1.25,
    centroid_min_hz=2200,
    centroid_max_hz=9000,
    # trimming
    end_db_drop=20,
    end_hold_ms=18,
    min_snare_ms=60,
    max_snare_ms=420,
    # mp3
    mp3_bitrate="320k",
    ffmpeg_path="ffmpeg",
    overwrite=True,
)

TQM | mp3 → find 1 snare → trim → export mp3: 100%|██████████████████| 75549/75549 [10:31:52<00:00,  1.99it/s]


# CUT the ones that are not one SNARES


In [7]:
# ===========================================
# -----######-----######  CORE FUNCTION  ######
# ===========================================

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

def _snare_0403_i1_GET_df_snare_SCORE_mp3(
    folder_path,
    sr=44100,

    # --- duration / shape (snare hits vary: short + medium tails) ---
    peak_must_be_within_ms=220,
    min_duration_ms=40,
    max_duration_ms=2200,

    # --- bands (snare = mid body + high "snap", NOT low-kick heavy) ---
    low_band_hz=(20, 160),          # reject kicks / toms
    body_band_hz=(170, 1200),       # snare body
    crack_band_hz=(2500, 9000),     # snap/crack
    air_band_hz=(9000, 16000),      # airy top (optional)

    # --- SINGLE HIT gates (loose-ish) ---
    max_onset_peaks=2,              # allow small double transient sometimes
    onset_peak_wait_ms=90,
    onset_peak_rel_thresh=0.50,
    max_env_peaks=2,
    env_peak_rel_thresh=0.52,

    # --- envelope sanity (loose) ---
    max_postpeak_rise_frac=0.34,
    retrigger_rms_rel=0.55,
    retrigger_min_ms=140,

    # --- texture (snare has noise, but can be more tonal than hats) ---
    min_spectral_flatness=0.10,
    min_zcr=0.030,

    # --- scoring (NOT too strict) ---
    score_keep_threshold=2.55,      # lower => keeps more snares
    score_weights=None,

    # --- file actions ---
    erase_mode="move",              # "move" (recommended) or "delete"
    quarantine_folder_name="_TRASH_NOT_SNARES",
    dry_run=True,
    audio_extensions=(".mp3",),
    verbose=False,
):
    """
    Snare detector using a weighted SCORE (intentionally NOT too strict).
    Keeps samples likely to be a snare/clap-ish hit (snare-heavy),
    removes non-snares immediately.

    Returns df with score + diagnostics.

    dry_run=True => no files moved/deleted, only reports.
    """

    import librosa

    if score_weights is None:
        # Snare identity = body + crack; low-end should NOT dominate
        score_weights = {
            "body_vs_low": 1.10,
            "crack_vs_low": 1.05,
            "crack_vs_body": 0.70,
            "flatness": 0.45,
            "zcr": 0.40,
            "attack": 0.45,
            "decay": 0.35,
        }

    def _is_bad_hidden_file(fn):
        return fn.startswith("._") or fn.startswith(".DS") or fn.startswith("._DS")

    def _hz_to_bin(hz, n_fft, sr_):
        return int(np.clip(np.round(hz * n_fft / sr_), 0, n_fft // 2))

    def _count_peaks_simple(x, rel_thresh=0.6, min_dist=4):
        x = np.asarray(x, dtype=float)
        if len(x) < 5:
            return 0
        mx = float(np.max(x)) + 1e-12
        thr = rel_thresh * mx

        peaks = []
        for i in range(1, len(x) - 1):
            if x[i] > thr and x[i] >= x[i-1] and x[i] >= x[i+1]:
                if not peaks or (i - peaks[-1]) >= min_dist:
                    peaks.append(i)
        return len(peaks)

    def _clip01(v):
        return float(np.clip(v, 0.0, 1.0))

    def _score_log_ratio(r, lo, hi):
        r = float(max(r, 1e-12))
        x = np.log10(r)
        return _clip01((x - lo) / (hi - lo + 1e-12))

    def _analyze_one(path_mp3):
        out = {
            "Path": path_mp3,
            "file_name": os.path.basename(path_mp3),
            "decision": "UNKNOWN",
            "reason": "",
            "dur_ms": np.nan,
            "peak_time_ms": np.nan,

            "onset_peaks": np.nan,
            "env_peaks": np.nan,
            "postpeak_rise_frac": np.nan,
            "retrigger_flag": np.nan,

            "flatness": np.nan,
            "zcr": np.nan,

            "body_low_ratio": np.nan,
            "crack_low_ratio": np.nan,
            "crack_body_ratio": np.nan,
            "air_crack_ratio": np.nan,

            "score": np.nan,
            "error": "",
        }

        try:
            y, sr_ = librosa.load(path_mp3, sr=sr, mono=True)
            if y is None or len(y) < 32:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "empty_audio"
                return out

            dur_ms = (len(y) / sr_) * 1000.0
            out["dur_ms"] = float(dur_ms)

            if dur_ms < min_duration_ms:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "too_short"
                return out
            if dur_ms > max_duration_ms:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "too_long"
                return out

            # normalize
            peak_abs = float(np.max(np.abs(y))) + 1e-12
            y = y / peak_abs

            # peak early-ish
            pk_i = int(np.argmax(np.abs(y)))
            pk_t_ms = (pk_i / sr_) * 1000.0
            out["peak_time_ms"] = float(pk_t_ms)
            if pk_t_ms > peak_must_be_within_ms:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "peak_too_late"
                return out

            # start at peak
            y_post = y[pk_i:]
            if len(y_post) < 1024:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "too_short_postpeak"
                return out

            hop = 256
            frame_len = 1024

            # ---- multi-hit gate (LOOSE) ----
            onset_env = librosa.onset.onset_strength(y=y_post, sr=sr_, hop_length=hop)
            if onset_env is None or len(onset_env) < 6:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "onset_failed"
                return out

            min_dist_frames = max(1, int((onset_peak_wait_ms / 1000.0) * sr_ / hop))
            onset_peaks = _count_peaks_simple(onset_env, rel_thresh=onset_peak_rel_thresh, min_dist=min_dist_frames)
            out["onset_peaks"] = int(onset_peaks)
            if onset_peaks > max_onset_peaks:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "too_many_onsets"
                return out

            rms = librosa.feature.rms(y=y_post, frame_length=frame_len, hop_length=hop, center=False)[0]
            if rms is None or len(rms) < 8:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "rms_failed"
                return out

            rms = np.maximum(rms, 1e-9)
            rms = rms / (np.max(rms) + 1e-12)

            env_peaks = _count_peaks_simple(rms, rel_thresh=env_peak_rel_thresh, min_dist=min_dist_frames)
            out["env_peaks"] = int(env_peaks)
            if env_peaks > max_env_peaks:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "too_many_env_peaks"
                return out

            diffs = np.diff(rms)
            rises = np.sum(diffs > 0)
            out["postpeak_rise_frac"] = float(rises / max(len(diffs), 1))
            if out["postpeak_rise_frac"] > max_postpeak_rise_frac:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "envelope_rises_too_much"
                return out

            # retrigger check (loose)
            t_ms = (np.arange(len(rms)) * hop / sr_) * 1000.0
            start_idx = int(np.argmax(t_ms >= retrigger_min_ms)) if np.any(t_ms >= retrigger_min_ms) else len(rms)
            retrigger_flag = 0
            if start_idx < len(rms):
                tail = rms[start_idx:]
                if len(tail) >= 4:
                    min_tail = float(np.min(tail))
                    max_tail = float(np.max(tail))
                    if (max_tail - min_tail) > 0.60 and max_tail > retrigger_rms_rel:
                        retrigger_flag = 1
            out["retrigger_flag"] = int(retrigger_flag)
            if retrigger_flag == 1:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "retrigger_detected"
                return out

            # ---- texture (loose) ----
            flat = librosa.feature.spectral_flatness(y=y_post)[0]
            flat_v = float(np.median(flat)) if flat is not None and len(flat) else 0.0
            out["flatness"] = float(flat_v)
            if flat_v < min_spectral_flatness:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "too_tonal"
                return out

            zcr = librosa.feature.zero_crossing_rate(y_post, frame_length=1024, hop_length=hop, center=False)[0]
            zcr_v = float(np.median(zcr)) if zcr is not None and len(zcr) else 0.0
            out["zcr"] = float(zcr_v)
            if zcr_v < min_zcr:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "zcr_too_low"
                return out

            # ---- spectral ratios ----
            cap = min(len(y_post), int(1.0 * sr_))   # snares can have tails
            y_cap = y_post[:cap] if cap > 512 else y_post

            n_fft = 4096
            S = np.abs(librosa.stft(y_cap, n_fft=n_fft, hop_length=hop, center=False)) ** 2
            if S is None or S.size == 0:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "stft_failed"
                return out

            lb0 = _hz_to_bin(low_band_hz[0], n_fft, sr_)
            lb1 = _hz_to_bin(low_band_hz[1], n_fft, sr_)
            bb0 = _hz_to_bin(body_band_hz[0], n_fft, sr_)
            bb1 = _hz_to_bin(body_band_hz[1], n_fft, sr_)
            cb0 = _hz_to_bin(crack_band_hz[0], n_fft, sr_)
            cb1 = _hz_to_bin(crack_band_hz[1], n_fft, sr_)
            ab0 = _hz_to_bin(air_band_hz[0], n_fft, sr_)
            ab1 = _hz_to_bin(air_band_hz[1], n_fft, sr_)

            e_low = float(np.sum(S[lb0:lb1 + 1, :])) + 1e-12
            e_body = float(np.sum(S[bb0:bb1 + 1, :])) + 1e-12
            e_crack = float(np.sum(S[cb0:cb1 + 1, :])) + 1e-12
            e_air = float(np.sum(S[ab0:ab1 + 1, :])) + 1e-12

            body_low = e_body / e_low
            crack_low = e_crack / e_low
            crack_body = e_crack / e_body
            air_crack = e_air / e_crack

            out["body_low_ratio"] = float(body_low)
            out["crack_low_ratio"] = float(crack_low)
            out["crack_body_ratio"] = float(crack_body)
            out["air_crack_ratio"] = float(air_crack)

            # ---- SCORE ----
            # body_low: snares tend to have more body than pure low-end
            s_body_low = _score_log_ratio(body_low, lo=0.05, hi=0.85)      # ~1.1x..7x
            # crack_low: even “dull” snares still have some crack
            s_crack_low = _score_log_ratio(crack_low, lo=-0.10, hi=0.75)   # ~0.8x..5.6x
            # crack_body: keeps claps/snare-ish; too low => thuddy tom
            s_crack_body = _score_log_ratio(crack_body, lo=-0.30, hi=0.55) # ~0.5x..3.5x

            s_flat = _clip01((flat_v - 0.10) / (0.34 - 0.10 + 1e-12))
            s_zcr  = _clip01((zcr_v - 0.03) / (0.14 - 0.03 + 1e-12))

            s_attack = _clip01((peak_must_be_within_ms - pk_t_ms) / peak_must_be_within_ms)
            s_decay = _clip01((0.42 - out["postpeak_rise_frac"]) / 0.42)

            score = (
                score_weights["body_vs_low"]   * s_body_low +
                score_weights["crack_vs_low"]  * s_crack_low +
                score_weights["crack_vs_body"] * s_crack_body +
                score_weights["flatness"]      * s_flat +
                score_weights["zcr"]           * s_zcr +
                score_weights["attack"]        * s_attack +
                score_weights["decay"]         * s_decay
            )
            out["score"] = float(score)

            if score >= score_keep_threshold:
                out["decision"] = "SNARE"
                out["reason"] = "passed_SCORE"
            else:
                out["decision"] = "NOT_SNARE"
                out["reason"] = "score_too_low"

            return out

        except Exception as e:
            out["decision"] = "NOT_SNARE"
            out["reason"] = "exception"
            out["error"] = str(e)
            return out

    folder_path = os.path.abspath(os.path.expanduser(str(folder_path)))
    if not os.path.isdir(folder_path):
        raise ValueError(f"folder_path not found: {folder_path}")

    exts_lower = tuple([e.lower() for e in audio_extensions])

    files = []
    for fn in os.listdir(folder_path):
        if _is_bad_hidden_file(fn):
            continue
        full = os.path.join(folder_path, fn)
        if os.path.isfile(full) and fn.lower().endswith(exts_lower):
            files.append(full)

    quarantine_dir = os.path.join(folder_path, quarantine_folder_name)
    if erase_mode == "move" and (not dry_run):
        os.makedirs(quarantine_dir, exist_ok=True)

    rows = []
    for p in tqdm(files, desc="SNARE SCORE — scanning mp3"):
        res = _analyze_one(p)
        rows.append(res)

        if res["decision"] == "NOT_SNARE" and (not dry_run):
            if erase_mode == "move":
                dest = os.path.join(quarantine_dir, os.path.basename(p))
                if os.path.exists(dest):
                    base, ext = os.path.splitext(os.path.basename(p))
                    k = 1
                    while True:
                        dest2 = os.path.join(quarantine_dir, f"{base}__dup{k}{ext}")
                        if not os.path.exists(dest2):
                            dest = dest2
                            break
                        k += 1
                shutil.move(p, dest)
            elif erase_mode == "delete":
                os.remove(p)

    df = pd.DataFrame(rows)
    df["is_snare"] = df["decision"].eq("SNARE")
    df["action"] = "KEEP"
    df.loc[df["decision"].eq("NOT_SNARE"), "action"] = ("DRY_RUN_SKIP" if dry_run else ("MOVE_TO_QUARANTINE" if erase_mode=="move" else "DELETE"))

    if verbose:
        n_all = len(df)
        n_s = int(df["is_snare"].sum())
        n_ns = n_all - n_s
        print("\n---- SNARE SCORE SUMMARY ----")
        print(f"folder: {folder_path}")
        print(f"mp3 scanned: {n_all}")
        print(f"KEEP (SNARE): {n_s}")
        print(f"ERASE (NOT_SNARE): {n_ns}")
        print(f"dry_run: {dry_run} | erase_mode: {erase_mode}")
        print(f"score_keep_threshold: {score_keep_threshold}")
        if erase_mode == "move":
            print(f"quarantine_dir: {quarantine_dir}")

    return df

In [8]:
folder_path = out_dir


# Preview (keeps MORE snares)
df_snares = _snare_0403_i1_GET_df_snare_SCORE_mp3(
    folder_path=folder_path,
    dry_run=True,
    erase_mode="move",
    score_keep_threshold=2.55,
    verbose=True
)

# If you still want to keep more:
# score_keep_threshold=2.30

# Execute
df_snares = _snare_0403_i1_GET_df_snare_SCORE_mp3(
    folder_path=folder_path,
    dry_run=False,
    erase_mode="move",
    score_keep_threshold=2.55,
    verbose=True
)

SNARE SCORE — scanning mp3:   0%|                                          | 34/28416 [00:01<30:03, 15.74it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1456
  warnings.warn(
SNARE SCORE — scanning mp3:   0%|                                        | 54/28416 [00:03<1:04:13,  7.36it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1034
  warnings.warn(
SNARE SCORE — scanning mp3:   0%|                                          | 68/28416 [00:05<44:39, 10.58it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1863
  warnings.warn(
SNARE SCORE — scanning mp3:   0%|▏                                         | 90/28416 [00:07<45:30, 10.37it/s]/User


---- SNARE SCORE SUMMARY ----
folder: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_2_SNARE
mp3 scanned: 28416
KEEP (SNARE): 0
ERASE (NOT_SNARE): 28416
dry_run: True | erase_mode: move
score_keep_threshold: 2.55
quarantine_dir: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_2_SNARE/_TRASH_NOT_SNARES


SNARE SCORE — scanning mp3:   0%|                                                   | 0/28416 [00:00<?, ?it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1456
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1034
  warnings.warn(
SNARE SCORE — scanning mp3:   0%|                                         | 59/28416 [00:00<00:48, 583.84it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1863
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1588
  warnings.warn(
/Users/yerik/_apple_lib/_b_envs/2ms_env/


---- SNARE SCORE SUMMARY ----
folder: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_2_SNARE
mp3 scanned: 28416
KEEP (SNARE): 0
ERASE (NOT_SNARE): 28416
dry_run: False | erase_mode: move
score_keep_threshold: 2.55
quarantine_dir: /Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_2_SNARE/_TRASH_NOT_SNARES


# organize SNARES in folders 

In [11]:
#==============================================================#
# 0_FNS
#==============================================================#

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import librosa


#------------------------------#
# helpers (no ASCII art)
#------------------------------#
def _safe_mkdir(path):
    os.makedirs(path, exist_ok=True)
    return path

def _is_junk_file(fn):
    return fn.startswith("._") or fn.startswith(".DS") or fn in [".DS_Store"]

def _list_audio_files_1level(root_folder, audio_extensions):
    audio_extensions = [e.lower().lstrip(".") for e in (audio_extensions or [])]
    out = []
    for fn in os.listdir(root_folder):
        if _is_junk_file(fn):
            continue
        p = os.path.join(root_folder, fn)
        if os.path.isfile(p):
            ext = os.path.splitext(fn)[1].lower().lstrip(".")
            if (not audio_extensions) or (ext in audio_extensions):
                out.append(p)
    return sorted(out)

def _pct(arr, q):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.nanpercentile(arr, q))

def _mad(arr, eps=1e-12):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    med = np.median(arr)
    return float(np.median(np.abs(arr - med)) + eps)

def _robust_z(x, med, mad, eps=1e-12):
    if not np.isfinite(x) or not np.isfinite(med) or not np.isfinite(mad):
        return 0.0
    return float((x - med) / (1.4826 * mad + eps))

def _safe_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.mean(x))

def _safe_std(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.std(x))

def _stft_mag(y, sr, hop_length, n_fft=2048):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    return S, freqs

def _band_energy_ratios(S, freqs, bands_hz):
    # bands_hz: list of (name, f_lo, f_hi)
    e = {}
    for name, f0, f1 in bands_hz:
        mask = (freqs >= f0) & (freqs < f1)
        e[name] = float(np.mean(S[mask, :])) if np.any(mask) else 0.0
    total = sum(e.values()) + 1e-12
    r = {f"r_{k}": float(v / total) for k, v in e.items()}
    return e, r

def _env_metrics(y, sr, hop_length, top_db):
    y_trim, idx = librosa.effects.trim(y, top_db=top_db)
    if y_trim.size < 2048:
        y_trim = y
        idx = (0, len(y))

    dur = float(len(y_trim) / sr)

    rms = librosa.feature.rms(y=y_trim, hop_length=hop_length)[0]
    if rms.size < 8:
        return {"error": "rms_too_small"}

    rms_db = 20 * np.log10(np.maximum(rms, 1e-12))
    peak_db = float(np.max(rms_db))
    peak_i = int(np.argmax(rms_db))
    post = rms_db[peak_i:]
    if post.size < 3:
        return {"error": "post_peak_too_small"}

    def _time_to_drop(db_drop):
        target = peak_db - db_drop
        hit = np.where(post <= target)[0]
        if hit.size == 0:
            return dur
        return float((int(hit[0]) * hop_length) / sr)

    t_drop_06 = _time_to_drop(6)
    t_drop_12 = _time_to_drop(12)
    t_drop_24 = _time_to_drop(24)

    # attack ratio: early window vs later window
    peak_t = (peak_i * hop_length) / sr
    def _frame(tsec):
        return int(np.clip(np.round((tsec * sr) / hop_length), 0, rms.size - 1))

    f0 = _frame(max(0.0, peak_t))
    f1 = _frame(peak_t + 0.03)
    f2 = _frame(peak_t + 0.18)

    e_early = float(np.mean(rms[f0:max(f0 + 1, f1 + 1)]))
    e_late  = float(np.mean(rms[min(f1, rms.size - 1):max(min(f1, rms.size - 1) + 1, f2 + 1)]))
    attack_ratio = float(e_early / (e_late + 1e-12))

    # tail slope (abrupt cut / choke-like behavior)
    n = post.size
    tail_start = int(max(1, np.floor(n * 0.85)))
    tail = post[tail_start:]
    if tail.size >= 3:
        x = np.arange(tail.size)
        tail_slope = float(np.polyfit(x, tail, 1)[0])  # dB per frame (more negative => steeper)
    else:
        tail_slope = 0.0

    return {
        "error": "",
        "dur_s": dur,
        "t_drop_06": t_drop_06,
        "t_drop_12": t_drop_12,
        "t_drop_24": t_drop_24,
        "attack_ratio": attack_ratio,
        "tail_slope_db_per_frame": tail_slope,
        "trim_i0": int(idx[0]),
        "trim_i1": int(idx[1]),
    }

def _voice_signatures(y, sr, hop_length):
    """
    Stronger “voice contamination” detector:
      - harmonic/percussive ratio
      - voiced fraction + stable f0 (yin)
      - chroma strength (tonal content)
      - spectral flatness (noise-likeness)
    """
    # HPSS
    y_h, y_p = librosa.effects.hpss(y)
    eh = float(np.mean(y_h**2))
    ep = float(np.mean(y_p**2))
    harm_ratio = float(eh / (ep + 1e-12))  # higher => more harmonic

    # YIN (typical voice-ish range)
    f0 = librosa.yin(y, fmin=70, fmax=450, sr=sr)
    f0 = np.asarray(f0, dtype=float)
    f0_ok = f0[np.isfinite(f0) & (f0 > 0)]
    voiced_frac = float(f0_ok.size / max(1, f0.size))

    if f0_ok.size >= max(12, int(0.18 * f0.size)):
        f0_med = float(np.nanmedian(f0_ok))
        f0_std = float(np.nanstd(f0_ok))
    else:
        f0_med = np.nan
        f0_std = np.nan

    # Tonal strength via chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    chroma_mean = np.mean(chroma, axis=1)
    chroma_peak = float(np.max(chroma_mean))
    chroma_spread = float(np.std(chroma_mean))
    chroma_peaky = float(chroma_peak / (chroma_spread + 1e-12))  # higher => strong tonal center

    return {
        "harm_ratio": harm_ratio,
        "voiced_frac": voiced_frac,
        "f0_med": f0_med,
        "f0_std": f0_std,
        "chroma_peaky": chroma_peaky,
    }

def _extract_snare_features_v2(path, sr_target, top_db, hop_length):
    y, sr = librosa.load(path, sr=sr_target, mono=True)
    if y.size < 2048:
        return {"error": "too_short_audio"}

    # envelope (trim gently)
    env = _env_metrics(y, sr, hop_length=hop_length, top_db=top_db)
    if env.get("error"):
        return env

    # focus analysis on trimmed audio (better consistency)
    i0, i1 = env["trim_i0"], env["trim_i1"]
    y_trim = y[i0:i1] if (i1 > i0 and i1 <= len(y)) else y
    if y_trim.size < 2048:
        y_trim = y

    # core spectral features
    S, freqs = _stft_mag(y_trim, sr, hop_length=hop_length, n_fft=2048)
    centroid = float(np.mean(librosa.feature.spectral_centroid(S=S, sr=sr)))
    rolloff  = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)))
    flatness = float(np.mean(librosa.feature.spectral_flatness(S=S)))
    contrast = float(np.mean(librosa.feature.spectral_contrast(S=S, sr=sr)))
    zcr      = float(np.mean(librosa.feature.zero_crossing_rate(y_trim, hop_length=hop_length)))

    # spectral flux (frame-to-frame change) => percussive/noisy hits tend to be higher
    S_norm = S / (np.sum(S, axis=0, keepdims=True) + 1e-12)
    flux = float(np.mean(np.sqrt(np.sum(np.diff(S_norm, axis=1)**2, axis=0) + 1e-12))) if S_norm.shape[1] >= 2 else 0.0

    # onset strength (snare-ish transient indicator)
    onset_env = librosa.onset.onset_strength(y=y_trim, sr=sr, hop_length=hop_length)
    onset_peak = float(np.max(onset_env)) if onset_env.size else 0.0
    onset_mean = float(np.mean(onset_env)) if onset_env.size else 0.0
    onset_peaky = float(onset_peak / (onset_mean + 1e-12))

    # MFCC rough “noisiness vs voice” hints
    mfcc = librosa.feature.mfcc(y=y_trim, sr=sr, n_mfcc=13, hop_length=hop_length)
    mfcc_1 = float(np.mean(mfcc[0])) if mfcc.shape[0] >= 1 else 0.0
    mfcc_2 = float(np.mean(mfcc[1])) if mfcc.shape[0] >= 2 else 0.0
    mfcc_var = float(np.mean(np.var(mfcc, axis=1))) if mfcc.size else 0.0

    # band energy ratios (snare signature is “mid noise” + some high snap, with optional low body)
    bands = [
        ("sub", 0, 120),
        ("body", 120, 400),
        ("lowmid", 400, 1800),
        ("highmid", 1800, 8000),
        ("air", 8000, 20000),
    ]
    _, ratios = _band_energy_ratios(S, freqs, bands)

    # voice signatures computed on trimmed
    voice = _voice_signatures(y_trim, sr=sr, hop_length=hop_length)

    return {
        "error": "",
        **env,
        "centroid_hz": centroid,
        "rolloff_hz": rolloff,
        "flatness": flatness,
        "contrast": contrast,
        "zcr": zcr,
        "flux": flux,
        "onset_peaky": onset_peaky,
        "onset_peak": onset_peak,
        "mfcc_1": mfcc_1,
        "mfcc_2": mfcc_2,
        "mfcc_var": mfcc_var,
        **ratios,
        **voice,
    }

def _build_stats(df_ok):
    """
    Robust stats for z-scores + adaptive percentiles.
    """
    cols = [
        "dur_s","t_drop_06","t_drop_12","t_drop_24","attack_ratio","tail_slope_db_per_frame",
        "centroid_hz","rolloff_hz","flatness","contrast","zcr","flux","onset_peaky",
        "r_sub","r_body","r_lowmid","r_highmid","r_air",
        "harm_ratio","voiced_frac","chroma_peaky","f0_med","f0_std",
        "mfcc_1","mfcc_2","mfcc_var"
    ]
    stats = {"pct": {}, "med": {}, "mad": {}}
    for c in cols:
        if c not in df_ok.columns:
            continue
        v = df_ok[c].astype(float).values
        stats["pct"][c] = {
            "p20": _pct(v, 20), "p35": _pct(v, 35), "p50": _pct(v, 50),
            "p65": _pct(v, 65), "p80": _pct(v, 80)
        }
        stats["med"][c] = float(np.nanmedian(v[np.isfinite(v)])) if np.any(np.isfinite(v)) else np.nan
        stats["mad"][c] = _mad(v)
    return stats

def _z(row, stats, col):
    return _robust_z(float(row.get(col, np.nan)), stats["med"].get(col, np.nan), stats["mad"].get(col, np.nan))

def _score_snare_classes(row, stats):
    """
    Score-based classifier (WAY finer than pure if/else):
    - computes class scores from robust z-features
    - returns (category, confidence, debug_str)
    """
    p = stats["pct"]

    # Core “snare hit” evidence (transient + mid noise)
    z_atk   = _z(row, stats, "attack_ratio")
    z_flux  = _z(row, stats, "flux")
    z_onpk  = _z(row, stats, "onset_peaky")
    z_mid   = _robust_z(float(row.get("r_lowmid",0)+row.get("r_highmid",0)), 
                        stats["med"].get("r_lowmid",0)+stats["med"].get("r_highmid",0),
                        _mad((row.get("r_lowmid",0)+row.get("r_highmid",0))), 1e-12)  # safe-ish
    z_flat  = _z(row, stats, "flatness")
    z_zcr   = _z(row, stats, "zcr")

    # Sustain / length
    z_t24   = _z(row, stats, "t_drop_24")
    z_dur   = _z(row, stats, "dur_s")

    # Brightness / snap
    z_cent  = _z(row, stats, "centroid_hz")
    z_air   = _z(row, stats, "r_air")

    # Low body (some snares have a “thump”)
    z_body  = _z(row, stats, "r_body")

    # Voice signatures
    z_harm  = _z(row, stats, "harm_ratio")
    z_voic  = _z(row, stats, "voiced_frac")
    z_chr   = _z(row, stats, "chroma_peaky")
    f0_med  = row.get("f0_med", np.nan)
    f0_ok   = 1.0 if (np.isfinite(f0_med) and 70 <= float(f0_med) <= 450) else 0.0

    # Gate: percussive likelihood (keeps obvious pads/loops out)
    percussive_evidence = (
        0.55*z_atk + 0.45*z_onpk + 0.45*z_flux + 0.35*z_zcr + 0.25*z_flat
    )

    # Gate: mid-noise snare signature
    mid_noise = (row.get("r_lowmid",0)+row.get("r_highmid",0))
    mid_noise_gate = 1.0 if (np.isfinite(mid_noise) and mid_noise >= p["r_highmid"]["p35"]) else 0.0

    # Voice score (stronger + more reliable)
    voice_score = (0.70*z_harm + 0.65*z_voic + 0.50*z_chr + 0.40*f0_ok - 0.35*z_flat)

    # Class scores
    scores = {}

    # Tight: short sustain, sharp transient
    scores["01_SNARE_TIGHT"] = (
        0.80*percussive_evidence + 0.55*z_mid + 0.35*z_cent + 0.20*z_air
        - 0.65*z_t24 - 0.35*z_dur
    )

    # Body: balanced sustain + some low body + mid noise
    scores["02_SNARE_BODY"] = (
        0.75*percussive_evidence + 0.60*z_mid + 0.35*z_body + 0.15*z_cent
        - 0.15*np.abs(z_t24)
    )

    # Long: longer sustain / tail
    scores["03_SNARE_LONG"] = (
        0.65*percussive_evidence + 0.45*z_mid + 0.35*z_t24 + 0.25*z_dur
    )

    # Rimshot: very short, very bright, low body
    scores["04_RIMSHOT"] = (
        0.85*percussive_evidence + 0.55*z_cent + 0.45*z_air
        - 0.55*z_t24 - 0.40*z_dur - 0.35*z_body
    )

    # Clap-like: wider transient + less low body + often lower mid focus (still percussive)
    # (this helps avoid claps polluting snares)
    scores["06_CLAP_LIKE"] = (
        0.70*percussive_evidence + 0.25*z_air + 0.20*z_cent
        - 0.35*z_mid - 0.30*z_body
    )

    # FX/Noise: super flat + high flux but weak snare shape / very long / weird
    scores["09_FX_NOISE"] = (
        0.55*z_flat + 0.55*z_flux + 0.25*z_dur - 0.30*z_mid - 0.20*z_body
    )

    # Voicey snare: both snare evidence + voice score
    scores["07_VOICEY_SNARE"] = (
        0.70*percussive_evidence + 0.35*z_mid + 0.25*z_cent + 0.75*voice_score
    )

    # Voice only: high voice score but weak percussive snare evidence
    scores["08_VOICE_ONLY"] = (
        1.15*voice_score - 0.55*percussive_evidence - 0.25*z_mid
    )

    # Not snare: inverse evidence
    scores["10_NOT_SNARE"] = (
        -0.85*percussive_evidence - 0.45*z_mid + 0.25*z_dur + 0.30*voice_score
    )

    # Pick best
    best = max(scores.items(), key=lambda kv: kv[1])
    cat, best_score = best[0], float(best[1])

    # Confidence: margin between top-2 (simple + reliable)
    top2 = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:2]
    margin = float(top2[0][1] - top2[1][1]) if len(top2) == 2 else 0.0

    # Guard rails:
    # - if voice_only is high and percussive evidence is low => voice only
    # - if mid_noise_gate is false and percussive evidence is weak => not snare / fx
    if voice_score >= 1.0 and percussive_evidence <= 0.3:
        cat = "08_VOICE_ONLY"
    if (mid_noise_gate < 0.5) and (percussive_evidence < 0.15) and (cat not in ["08_VOICE_ONLY","07_VOICEY_SNARE"]):
        cat = "10_NOT_SNARE"

    # confidence scaled 0..1 from margin (tunable)
    conf = float(1.0 / (1.0 + np.exp(-2.2 * margin)))  # logistic of margin

    dbg = f"best={best_score:.3f} margin={margin:.3f} perc={percussive_evidence:.3f} voice={voice_score:.3f}"
    return cat, conf, dbg


#==============================================================#
#-----######-----######  CORE IMPORTABLE FUNCTION  ######-----##
#==============================================================#
def _snare_0503_i2_GET_df_sorted_snares(
    folder_snares,
    audio_extensions,
    sr_target=44100,
    top_db=35,
    hop_length=256,
    make_folders=True,
    move_files=True,
    dry_run=False,
    keep_unsure=True,
    min_confidence=0.62,     # anything below goes to UNSURE (super useful)
    use_subfolders=True,     # True => nested folder layout (cleaner)
):
    """
    SUPER fine snare sorter with voice contamination detection.

    Categories (created inside the SAME folder):
      01_SNARE_TIGHT
      02_SNARE_BODY
      03_SNARE_LONG
      04_RIMSHOT
      06_CLAP_LIKE
      07_VOICEY_SNARE
      08_VOICE_ONLY
      09_FX_NOISE
      10_NOT_SNARE
      UNSURE (optional)

    If use_subfolders=True, it will create:
      SNARES/01_SNARE_TIGHT, etc
    otherwise it creates those folders directly in folder_snares.

    Returns df with category + confidence + debug.
    """
    folder_snares = os.path.abspath(folder_snares)
    files = _list_audio_files_1level(folder_snares, audio_extensions)

    if not files:
        return pd.DataFrame([{
            "Path": "",
            "file_name": "",
            "category": "",
            "confidence": np.nan,
            "dest_path": "",
            "error": f"No audio files found in: {folder_snares}"
        }])

    rows = []
    for pth in tqdm(files, desc="Analyzing snares (fine)", total=len(files)):
        fn = os.path.basename(pth)
        d = {"Path": pth, "file_name": fn}
        try:
            feats = _extract_snare_features_v2(pth, sr_target=sr_target, top_db=top_db, hop_length=hop_length)
            d.update(feats)
        except Exception as e:
            d.update({"error": f"extract_fail: {type(e).__name__}: {e}"})
        rows.append(d)

    df = pd.DataFrame(rows)

    df_ok = df[df["error"].fillna("") == ""].copy()
    if df_ok.empty:
        df["category"] = "UNSURE"
        df["confidence"] = 0.0
        df["debug"] = "no_valid_rows"
    else:
        stats = _build_stats(df_ok)

        cats = []
        confs = []
        debugs = []
        for _, r in df.iterrows():
            if str(r.get("error","")):
                cats.append("UNSURE")
                confs.append(0.0)
                debugs.append("error_row")
            else:
                c, conf, dbg = _score_snare_classes(r, stats)
                # low confidence => UNSURE bucket (prevents bad moves)
                if keep_unsure and (conf < float(min_confidence)):
                    c = "UNSURE"
                cats.append(c)
                confs.append(conf)
                debugs.append(dbg)

        df["category"] = cats
        df["confidence"] = confs
        df["debug"] = debugs

    buckets = [
        "01_SNARE_TIGHT",
        "02_SNARE_BODY",
        "03_SNARE_LONG",
        "04_RIMSHOT",
        "06_CLAP_LIKE",
        "07_VOICEY_SNARE",
        "08_VOICE_ONLY",
        "09_FX_NOISE",
        "10_NOT_SNARE",
    ]
    if keep_unsure:
        buckets += ["UNSURE"]

    # root for folders
    root_out = folder_snares
    if use_subfolders:
        root_out = _safe_mkdir(os.path.join(folder_snares, "SNARES_SORTED"))

    if make_folders:
        for b in buckets:
            _safe_mkdir(os.path.join(root_out, b))

    # destination + move (with TQDM for moving too)
    dest_paths = []
    for i, r in tqdm(df.iterrows(), desc="Moving files", total=len(df)):
        cat = r["category"]
        if cat not in buckets:
            cat = "UNSURE" if keep_unsure else "10_NOT_SNARE"

        src = r["Path"]
        fn = r["file_name"]
        dest_dir = os.path.join(root_out, cat)
        dest = os.path.join(dest_dir, fn)

        # avoid overwrite
        if os.path.exists(dest) and (os.path.abspath(src) != os.path.abspath(dest)):
            base, ext = os.path.splitext(fn)
            k = 2
            while True:
                cand = os.path.join(dest_dir, f"{base}_v{k}{ext}")
                if not os.path.exists(cand):
                    dest = cand
                    break
                k += 1

        dest_paths.append(dest)

        if move_files and (not dry_run):
            # if already correct, skip
            if os.path.dirname(os.path.abspath(src)) == os.path.abspath(dest_dir):
                continue
            try:
                shutil.move(src, dest)
            except Exception as e:
                df.loc[df["Path"] == src, "error"] = (
                    df.loc[df["Path"] == src, "error"].astype(str) + f" | move_fail: {e}"
                ).values

    df["dest_path"] = dest_paths

    cols_front = ["file_name", "category", "confidence", "Path", "dest_path", "error", "debug"]
    cols_rest = [c for c in df.columns if c not in cols_front]
    df = df[cols_front + cols_rest].sort_values(["category", "confidence", "file_name"], ascending=[True, False, True]).reset_index(drop=True)

    return df

In [12]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

folder_snares = out_dir
audio_extensions = ["mp3", "wav", "aiff", "aif", "flac", "m4a"]

df_snares = _snare_0503_i2_GET_df_sorted_snares(
    folder_snares=folder_snares,
    audio_extensions=audio_extensions,
    sr_target=44100,
    top_db=35,
    hop_length=256,
    make_folders=True,
    move_files=True,
    dry_run=False,          # True = test, no moving
    keep_unsure=True,
    min_confidence=0.62,    # raise to 0.70 if you want fewer wrong moves
    use_subfolders=True     # puts everything in SNARES_SORTED/
)

Analyzing snares (fine):   1%|▎                                           | 164/28413 [00:04<10:21, 45.45it/s]/Users/yerik/_apple_lib/_b_envs/2ms_env/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
Moving files: 100%|███████████████████████████████████████████████████| 28413/28413 [00:05<00:00, 5660.16it/s]


In [ ]:
# END 
print("SNARES - DONE")